# Early Detection of Chronic Cardiometabolic Disease: End-to-End Clinical ML Pipeline

**Author**: Healthcare ML Research Team  
**Clinical Standard**: ADA & AHA Guideline-Aligned  
**Objective**: Early 1-year prediction of Type 2 Diabetes, Essential Hypertension, and Renal Complications from longitudinal EHR time-series and tabular demographic encounters.

---

## Notebook Overview
1. **Cohort Ingestion & Exploration**: Multi-modal longitudinal EHR data (1,000 patients, 4,995 visits).
2. **Feature Engineering**: Static demographic aggregations vs. 3D padded time-series tensors.
3. **Three-Way Clinical Modeling**:
   - Fast Tabular Baseline: Calibrated XGBoost with class imbalance optimization (`scale_pos_weight`)
   - Sequence-Aware Deep Model: PyTorch Bidirectional GRU with Masked Temporal Attention
   - Clinical Dual-Attention Model: PyTorch RETAIN (Reverse Time Attention) for encounter-level and variable-level interpretability
4. **Clinical Decision Curve Analysis (DCA)**: Net Benefit & unnecessary interventions avoided per 100 patients.
5. **Interpretability Suite**: Global SHAP beeswarm, patient waterfall attributions, and RETAIN temporal attention heatmaps.

In [ ]:
import os
import sys
from pathlib import Path
import json

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image

print(f"Project Root: {PROJECT_ROOT}")
print(f"PyTorch Version: {torch.__version__}")

## 1. Load Processed Cohort & Check Patient Leakage
We verify that our stratified patient split contains strictly zero patient overlap between train and test cohorts.

In [ ]:
data_dir = PROJECT_ROOT / "data" / "processed"
df_train = pd.read_csv(data_dir / "train_tabular.csv")
df_test = pd.read_csv(data_dir / "test_tabular.csv")

train_pts = set(df_train["patient_id"])
test_pts = set(df_test["patient_id"])
overlap = train_pts.intersection(test_pts)

target_col = "target_label" if "target_label" in df_train.columns else "target_disease"
prevalence_train = df_train[target_col].mean()
prevalence_test = df_test[target_col].mean()

print(f"Train cohort: {len(df_train)} patients | Disease Prevalence: {prevalence_train:.1%}")
print(f"Test cohort:  {len(df_test)} patients | Disease Prevalence: {prevalence_test:.1%}")
print(f"Patient overlap between splits: {len(overlap)} (Zero Leakage: {len(overlap) == 0})")

## 2. Benchmark Comparison Leaderboard
We load the evaluation metrics across the three models trained on the held-out test cohort.

In [ ]:
from src.tracking.experiment_tracker import ChronicDiseaseExperimentTracker

tracker = ChronicDiseaseExperimentTracker()
leaderboard = tracker.generate_leaderboard()
display(leaderboard)

## 3. Decision Curve Analysis (DCA) & Clinical Net Benefit
In clinical medicine, AUROC alone does not quantify patient harm. Decision Curve Analysis computes the **Net Benefit** across clinical risk thresholds $p_t$, assessing whether model-guided triage prevents unnecessary biopsies, medication side-effects, or excessive clinic visits.

In [ ]:
dca_plot_path = PROJECT_ROOT / "reports" / "decision_curve_analysis.png"
if dca_plot_path.exists():
    display(Image(filename=str(dca_plot_path)))
else:
    from src.evaluation.dca import ClinicalDecisionCurveAnalyzer
    analyzer = ClinicalDecisionCurveAnalyzer()
    analyzer.run_analysis()
    display(Image(filename=str(dca_plot_path)))

## 4. SHAP Interpretability: Global Risk Drivers & Patient Waterfall
Shapley Additive Explanations (SHAP) provide game-theoretic attributions for each clinical parameter.

In [ ]:
beeswarm_path = PROJECT_ROOT / "reports" / "shap_summary_beeswarm.png"
waterfall_path = PROJECT_ROOT / "reports" / "shap_patient_waterfall.png"

if beeswarm_path.exists():
    print("--- Global SHAP Summary (Cohort Beeswarm) ---")
    display(Image(filename=str(beeswarm_path)))

if waterfall_path.exists():
    print("--- Local Patient Risk Attribution (Waterfall) ---")
    display(Image(filename=str(waterfall_path)))

## 5. RETAIN Reverse-Time Dual Attention Interpretability
RETAIN decomposes patient trajectory into visit attention $\alpha_t$ and biomarker contribution $\beta_t$.

In [ ]:
from src.models.retain_model import RETAINSequenceTrainer

retain_path = PROJECT_ROOT / "models" / "retain_sequence_model.pt"
trainer = RETAINSequenceTrainer.load_checkpoint(retain_path)

# Inspect normalization statistics
print(f"RETAIN Feature Dimension: {len(trainer.feature_names)}")
print(f"Key Biomarkers Monitored: {trainer.feature_names[:10]}")

## 6. Clinical Distribution Shift & Population Stability (PSI)
We evaluate whether the monitored test cohort exhibits covariate drift against the training baseline.

In [ ]:
from src.monitoring.drift_detector import ClinicalDriftDetector

detector = ClinicalDriftDetector()
drift_report = detector.evaluate_cohort(df_test)

print(f"Cohort Safety Code: {drift_report['cohort_safety_code']}")
print(f"Cohort Status:      {drift_report['cohort_status']}")
print(f"Max Feature PSI:    {drift_report['max_psi']:.4f}")

df_drift = pd.DataFrame(drift_report["features"])
display(df_drift[["feature", "psi", "ks_statistic", "ks_pvalue", "drift_tier"]])